# Affine-backend reproduction notebook for arXiv:2603.06431 (Lp + W1p only)

This notebook mirrors the paper-style experiments for **Lp** and **W1p** using the **affine** domain backend, and intentionally excludes **W2p**.

⚠️ Stability note: this version uses **chunked Monte Carlo** for W1p and configurable quick settings to avoid kernel OOM/kill.


In [ ]:
# Notebook step 1: run the example/test logic for this section.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
while not (repo_root / "src" / "intervalnets").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

import gc
import math
import os
import random
from dataclasses import dataclass

# Workaround for OpenMP duplicate-runtime kernel crashes in some Torch/Matplotlib envs.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")

import numpy as np
import torch
from torch import nn
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
from pathlib import Path as _Path

print(f"matplotlib backend: {matplotlib.get_backend()}")

from IPython.display import Image, display

from intervalnets import AffineTensor, affine_forward, enable_interval_eval

# Force Chebyshev tanh relaxation for affine function-value and Sobolev/Jacobian AA workflows.
ENABLE_INTERVAL_EVAL_KWARGS = {"enclosure_mode": "slope", "affine_tanh_mode": "chebyshev"}
enable_interval_eval(**ENABLE_INTERVAL_EVAL_KWARGS)
torch.set_num_threads(max(1, min(torch.get_num_threads(), 8)))

print("\n=== Configuration summary [AA] ===")
print(f"domain type: {AffineTensor.__name__}")
print(
    "enable_interval_eval settings: "
    f"enclosure_mode={ENABLE_INTERVAL_EVAL_KWARGS.get('enclosure_mode', 'None')}, "
    f"affine_tanh_mode={ENABLE_INTERVAL_EVAL_KWARGS.get('affine_tanh_mode', 'None')}"
)
print("norm paths:")
print(f"  lp  -> model.lpnorm(..., domain={AffineTensor.__name__})")
print(f"  w1p -> model.sobolev_norm(..., domain={AffineTensor.__name__})")


## Configuration

Short description of the experiment/test performed in the following code cell.


In [ ]:
BASE_SEED = 1234

# Keep QUICK_MODE=True by default for stability in notebooks.
QUICK_MODE = False 

if QUICK_MODE:
    N_RUNS = 8
    ITERATIONS = list(range(0, 7))
    EPOCHS_1D = 150
    EPOCHS_2D_LP = 200
    EPOCHS_2D_W1P = 300
    MC_REF_SAMPLES_1D = 8_000
    MC_REF_SAMPLES_2D = 10_000
    MC_BATCH = 512
    DEEP_WIDTH = 10
    WIDE_WIDTH = 100
else:
    # Paper-like heavier settings
    N_RUNS = 100
    ITERATIONS = list(range(0, 30))[::5]
    EPOCHS_1D = 2000
    EPOCHS_2D_LP = 2000
    EPOCHS_2D_W1P = 10_000
    MC_REF_SAMPLES_1D = 50_000
    MC_REF_SAMPLES_2D = 50_000
    MC_BATCH = 2048
    DEEP_WIDTH = 32
    WIDE_WIDTH = 200

P_VAL = 2.0
print(f"QUICK_MODE={QUICK_MODE}, runs={N_RUNS}, iterations={len(ITERATIONS)}")
print(f"widths: deep=3x{DEEP_WIDTH}, wide=1x{WIDE_WIDTH}")
# `fill_between` can crash some notebook backends after heavy Torch workloads.
# Use "lines" by default for maximum kernel stability; set to "band" if your backend is stable.
PLOT_CI_STYLE = "lines"  # choices: "lines", "band"
PLOT_OUTPUT_DIR = _Path("notebooks") / "artifacts"
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)



### Backend note
This notebook assumes the non-interactive `Agg` backend to avoid renderer crashes with some Torch+Jupyter setups. Run the import cell first in a fresh kernel; if the printed backend is not `Agg`, restart the kernel and rerun from the top.

If your environment still crashes due to OpenMP duplicate runtime issues, this notebook also sets `KMP_DUPLICATE_LIB_OK=TRUE` before importing Torch.

Affine backend limitations/assumptions (current implementation):
- Supported layers follow the affine propagation implemented in `src/intervalnets/pytorch.py` (Sequential stacks and the core activation/linear operators used in this notebook).
- Refinement still partitions the concrete input box and hulls sub-results; with affine inputs, each partition is re-embedded as an affine box before evaluation.
- Bounds remain outward/conservative after concretization, so affine enclosures can still be numerically pessimistic for highly nonlinear regions.


## Helpers

Short description of the experiment/test performed in the following code cell.


In [ ]:
# Notebook step 6: run the example/test logic for this section.
@dataclass
class Arch:
    name: str
    input_dim: int
    hidden_layers: int
    width: int
    activation: str


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def make_network(arch: Arch) -> nn.Sequential:
    act = nn.Tanh if arch.activation == "tanh" else nn.ReLU
    layers = []
    in_dim = arch.input_dim
    for _ in range(arch.hidden_layers):
        layers.append(nn.Linear(in_dim, arch.width))
        layers.append(act())
        in_dim = arch.width
    layers.append(nn.Linear(in_dim, 1))
    return nn.Sequential(*layers)


def ci95(x: np.ndarray):
    m = x.mean(axis=0)
    if x.shape[0] <= 1:
        return m, m, m
    s = x.std(axis=0, ddof=1)
    h = 1.96 * s / np.sqrt(x.shape[0])
    return m, m - h, m + h


def sanitize_for_log(arr: np.ndarray, floor: float = 1e-14, ceil: float = 1e14) -> np.ndarray:
    arr = np.asarray(arr, dtype=float)
    arr = np.nan_to_num(arr, nan=ceil, posinf=ceil, neginf=floor)
    return np.clip(arr, floor, ceil)




def compute_plot_ci(arr: np.ndarray):
    m, lo, hi = ci95(arr)
    m = np.ascontiguousarray(sanitize_for_log(m), dtype=np.float64)
    lo = np.ascontiguousarray(sanitize_for_log(lo), dtype=np.float64)
    hi = np.ascontiguousarray(sanitize_for_log(hi), dtype=np.float64)

    # Enforce a valid ordering for plotting on log scale.
    lo = np.minimum(lo, m)
    hi = np.maximum(hi, m)
    hi = np.maximum(hi, lo * (1.0 + 1e-12))
    return m, lo, hi


def plot_ci_curve(ax, iterations, arr, label: str, color: str, ci_style: str = "lines"):
    x = np.ascontiguousarray(np.asarray(iterations, dtype=np.float64))
    m, lo, hi = compute_plot_ci(arr)
    ax.plot(x, m, color=color, label=label, linewidth=2.0)

    if ci_style == "band":
        # Some backends crash in `fill_between`; keep an explicit switch.
        ax.fill_between(x, lo, hi, color=color, alpha=0.2)
    else:
        ax.plot(x, lo, color=color, alpha=0.35, linestyle="--", linewidth=1.0)
        ax.plot(x, hi, color=color, alpha=0.35, linestyle="--", linewidth=1.0)




def finalize_figure(fig, filename: str, show_inline: bool = True):
    out_path = PLOT_OUTPUT_DIR / filename
    fig.savefig(out_path, dpi=160, bbox_inches="tight")
    print(f"saved figure: {out_path}")
    if show_inline:
        display(Image(filename=str(out_path)))
    plt.close(fig)

def gaussian_peak_1d(x: torch.Tensor) -> torch.Tensor:
    return torch.exp(-40.0 * x.pow(2))


def smooth_disk_2d(xy: torch.Tensor, radius: float = 0.6) -> torch.Tensor:
    r2 = xy[:, 0].pow(2) + xy[:, 1].pow(2)
    out = torch.zeros_like(r2)
    inside = r2 < radius * radius
    t = 1.0 - r2[inside] / (radius * radius)
    out[inside] = torch.exp(-1.0 / torch.clamp(t, min=1e-8))
    return out


def train_to_target(model: nn.Module, dim: int, target_fn, epochs: int, lr: float = 1e-3, batch_size: int = 1024):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for _ in range(epochs):
        x = torch.rand(batch_size, dim) * 2.0 - 1.0
        y = target_fn(x if dim > 1 else x[:, :1])
        pred = model(x).squeeze(-1)
        loss = ((pred - y) ** 2).mean()
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()


def mc_lp(model: nn.Module, dim: int, p: float, n: int, batch: int) -> float:
    model.eval()
    total = 0.0
    seen = 0
    with torch.no_grad():
        while seen < n:
            m = min(batch, n - seen)
            x = torch.rand(m, dim) * 2.0 - 1.0
            y = model(x).squeeze(-1).abs().pow(p)
            total += float(y.sum().item())
            seen += m
    integral = (2.0 ** dim) * total / n
    return integral ** (1.0 / p)


def mc_w1p(model: nn.Module, dim: int, p: float, n: int, batch: int) -> float:
    # Chunked to prevent kernel OOM from huge requires_grad tensors.
    model.eval()
    total = 0.0
    seen = 0
    while seen < n:
        m = min(batch, n - seen)
        x = torch.rand(m, dim, requires_grad=True) * 2.0 - 1.0
        y = model(x).squeeze(-1)
        grad = torch.autograd.grad(y.sum(), x, create_graph=False, retain_graph=False)[0]
        integrand = y.abs().pow(p) + torch.linalg.vector_norm(grad, ord=2, dim=-1).pow(p)
        total += float(integrand.detach().sum().item())
        seen += m
        del x, y, grad, integrand
    integral = (2.0 ** dim) * total / n
    return integral ** (1.0 / p)


def bound_gap_curve(model: nn.Module, domain: AffineTensor, p: float, mode: str, iterations: list[int], ref_value: float) -> np.ndarray:
    out = []
    for it in iterations:
        if mode == "lp":
            b = model.lpnorm(domain, p=p, iterations=it)
        else:
            b = model.sobolev_norm(domain, p=p, iterations=it)
        out.append((float(b.upper - b.lower)) / max(ref_value, 1e-12))
    return np.array(out, dtype=float)


## 1D setups

Short description of the experiment/test performed in the following code cell.


In [ ]:
# Notebook step 8: run the example/test logic for this section.
deep_tanh_1d = Arch("deep", 1, 3, DEEP_WIDTH, "tanh")
wide_tanh_1d = Arch("wide", 1, 1, WIDE_WIDTH, "tanh")

deep_relu_1d = Arch("deep", 1, 3, DEEP_WIDTH, "relu")
wide_relu_1d = Arch("wide", 1, 1, WIDE_WIDTH, "relu")

domain_1d = AffineTensor.from_bounds([-1.0], [1.0])


def run_family(arch: Arch, mode: str, trained: bool):
    curves = []
    for run in range(N_RUNS):
        set_seed(BASE_SEED + run)
        model = make_network(arch)
        if trained:
            train_to_target(model, dim=1, target_fn=gaussian_peak_1d, epochs=EPOCHS_1D)

        if mode == "w1p":
            ref = mc_w1p(model, dim=1, p=P_VAL, n=MC_REF_SAMPLES_1D, batch=MC_BATCH)
        else:
            ref = mc_lp(model, dim=1, p=P_VAL, n=MC_REF_SAMPLES_1D, batch=MC_BATCH)

        curves.append(bound_gap_curve(model, domain_1d, p=P_VAL, mode=mode, iterations=ITERATIONS, ref_value=ref))
        del model
        gc.collect()

    return np.stack(curves, axis=0)


## Figure A [AA] — 1D W1p (untrained vs trained)

Short description of the experiment/test performed in the following code cell.


In [ ]:
# Notebook step 10: run the example/test logic for this section.
w1p_deep_untrained = run_family(deep_tanh_1d, mode="w1p", trained=False)
w1p_wide_untrained = run_family(wide_tanh_1d, mode="w1p", trained=False)

w1p_deep_trained = run_family(deep_tanh_1d, mode="w1p", trained=True)
w1p_wide_trained = run_family(wide_tanh_1d, mode="w1p", trained=True)

plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], w1p_deep_untrained, w1p_wide_untrained, "[AA] Untrained tanh networks"),
    (axes[1], w1p_deep_trained, w1p_wide_trained, "[AA] Trained tanh networks (Gaussian peak)"),
]:
    for label, arr, color in [(f"deep (3x{DEEP_WIDTH})", deep_arr, "tab:blue"), (f"wide (1x{WIDE_WIDTH})", wide_arr, "tab:orange")]:
        plot_ci_curve(ax, ITERATIONS, arr, label=label, color=color, ci_style=PLOT_CI_STYLE)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("[AA] 1D W1p reproduction")
plt.tight_layout()
finalize_figure(fig, "aa_figure_a_w1p_1d.png")

## Figure B [AA] — 1D Lp (untrained vs trained)

Short description of the experiment/test performed in the following code cell.


In [ ]:
# Notebook step 12: run the example/test logic for this section.
lp_deep_untrained = run_family(deep_relu_1d, mode="lp", trained=False)
lp_wide_untrained = run_family(wide_relu_1d, mode="lp", trained=False)

lp_deep_trained = run_family(deep_relu_1d, mode="lp", trained=True)
lp_wide_trained = run_family(wide_relu_1d, mode="lp", trained=True)

plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, deep_arr, wide_arr, title in [
    (axes[0], lp_deep_untrained, lp_wide_untrained, "[AA] Untrained ReLU networks"),
    (axes[1], lp_deep_trained, lp_wide_trained, "[AA] Trained ReLU networks (Gaussian peak)"),
]:
    for label, arr, color in [(f"deep (3x{DEEP_WIDTH})", deep_arr, "tab:green"), (f"wide (1x{WIDE_WIDTH})", wide_arr, "tab:red")]:
        plot_ci_curve(ax, ITERATIONS, arr, label=label, color=color, ci_style=PLOT_CI_STYLE)
    ax.set_yscale("log")
    ax.set_xlabel("refinement iterations")
    ax.grid(True, alpha=0.3)
    ax.set_title(title)

axes[0].set_ylabel("normalized global bound gap")
axes[0].legend()
fig.suptitle("[AA] 1D Lp reproduction")
plt.tight_layout()
finalize_figure(fig, "aa_figure_b_lp_1d.png")

## 2D trained experiments [AA] (Figure C + D)

Short description of the experiment/test performed in the following code cell.


In [ ]:
# Notebook step 14: run the example/test logic for this section.
deep_relu_2d = Arch("deep", 2, 3, DEEP_WIDTH, "relu")
wide_relu_2d = Arch("wide", 2, 1, WIDE_WIDTH, "relu")
deep_tanh_2d = Arch("deep", 2, 3, DEEP_WIDTH, "tanh")
wide_tanh_2d = Arch("wide", 2, 1, WIDE_WIDTH, "tanh")
domain_2d = AffineTensor.from_bounds([-1.0, -1.0], [1.0, 1.0])

set_seed(BASE_SEED + 1000)
lp_deep_2d = make_network(deep_relu_2d)
train_to_target(lp_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP)
lp_ref_deep = mc_lp(lp_deep_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
lp_curve_deep = bound_gap_curve(lp_deep_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_deep)

set_seed(BASE_SEED + 1001)
lp_wide_2d = make_network(wide_relu_2d)
train_to_target(lp_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_LP)
lp_ref_wide = mc_lp(lp_wide_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
lp_curve_wide = bound_gap_curve(lp_wide_2d, domain_2d, p=P_VAL, mode="lp", iterations=ITERATIONS, ref_value=lp_ref_wide)

set_seed(BASE_SEED + 1100)
w1p_deep_2d = make_network(deep_tanh_2d)
train_to_target(w1p_deep_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P)
w1_ref_deep = mc_w1p(w1p_deep_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
w1_curve_deep = bound_gap_curve(w1p_deep_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1_ref_deep)

set_seed(BASE_SEED + 1101)
w1p_wide_2d = make_network(wide_tanh_2d)
train_to_target(w1p_wide_2d, dim=2, target_fn=smooth_disk_2d, epochs=EPOCHS_2D_W1P)
w1_ref_wide = mc_w1p(w1p_wide_2d, dim=2, p=P_VAL, n=MC_REF_SAMPLES_2D, batch=MC_BATCH)
w1_curve_wide = bound_gap_curve(w1p_wide_2d, domain_2d, p=P_VAL, mode="w1p", iterations=ITERATIONS, ref_value=w1_ref_wide)

plt.close("all")
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
axes[0].plot(ITERATIONS, lp_curve_deep, marker='o', label=f'deep (3x{DEEP_WIDTH})')
axes[0].plot(ITERATIONS, lp_curve_wide, marker='o', label=f'wide (1x{WIDE_WIDTH})')
axes[0].set_title('[AA] 2D trained Lp (ReLU)')
axes[0].set_yscale('log')
axes[0].set_xlabel('refinement iterations')
axes[0].set_ylabel('normalized global bound gap')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].plot(ITERATIONS, w1_curve_deep, marker='o', label=f'deep (3x{DEEP_WIDTH})')
axes[1].plot(ITERATIONS, w1_curve_wide, marker='o', label=f'wide (1x{WIDE_WIDTH})')
axes[1].set_title('[AA] 2D trained W1p (tanh)')
axes[1].set_yscale('log')
axes[1].set_xlabel('refinement iterations')
axes[1].grid(True, alpha=0.3)
axes[1].legend()
plt.tight_layout(); finalize_figure(fig, "aa_figure_cd_2d_curves.png")


def local_gap_heatmap(model: nn.Module, mode: str, grid_n: int = 20):
    xs = np.linspace(-1, 1, grid_n + 1)
    ys = np.linspace(-1, 1, grid_n + 1)
    out = np.zeros((grid_n, grid_n))
    for i in range(grid_n):
        for j in range(grid_n):
            box = AffineTensor.from_bounds([float(xs[i]), float(ys[j])], [float(xs[i+1]), float(ys[j+1])])
            b = model.lpnorm(box, p=P_VAL, iterations=0) if mode == 'lp' else model.sobolev_norm(box, p=P_VAL, iterations=0)
            out[j, i] = float(b.upper - b.lower)
    return out

h_lp = local_gap_heatmap(lp_deep_2d, 'lp')
h_w1 = local_gap_heatmap(w1p_deep_2d, 'w1p')
fig, axs = plt.subplots(1,2,figsize=(10,4))
axs[0].imshow(h_lp, origin='lower', extent=[-1,1,-1,1], cmap='magma'); axs[0].set_title('[AA] Lp local gap (deep)')
axs[1].imshow(h_w1, origin='lower', extent=[-1,1,-1,1], cmap='magma'); axs[1].set_title('[AA] W1p local gap (deep)')
plt.tight_layout(); finalize_figure(fig, "aa_figure_d_local_gap_heatmaps.png")

## Notes
Small description of the next experiment step.
